# H2-1 extension, step 6: main regression (household and week fixed effects + type x tier + clustered SE)

**Summary of the previous steps**
- Step 1: tiers fixed on weeks 17-32 (team's stable-window rule), 3 tiers (low/medium/high) with 834/834/832 households
- Steps 2-3: campaign type x week mapping, household x week panel (weeks 33-101, 172,500 rows)
- Step 4: type x tier cell sizes; only low x TypeC = 28 households is borderline, the rest are stable.
  Selection-bias check (reach 25.2% for low vs 94.1% for high)
- VIF check: VIF around 1.0 among the type-active dummies, correlations around 0.1, so no multicollinearity problem
- Step 5: parallel pre-trend test: **only the low tier passed cleanly (p=0.764)**,
  medium (p=0.053) and the full sample (p=0.087) are borderline, and high (p=0.613) has little power with only 49 non-recipients

**What this step does**
`spend ~ sum of 9 (Type x tier3) interaction dummies + household fixed effects + week fixed effects`,
with standard errors **clustered by household** (to handle autocorrelation in a panel of 84 repeated weeks).

**To be read together with the results (stated in advance)**
- Low-tier results: relatively the most reliable (passed step 5 + enough households per cell)
- Medium-tier results: campaign effect and pre-trend difference are not fully separated
- High-tier results: parallel pre-trend test has little power, interpret with care
- Low x TypeC: 28 households, borderline (only 2 floor-effect households, so the risk of double distortion is low)

## 0. Environment setup

In [1]:
from importlib.metadata import version
print("linearmodels", version("linearmodels"), "| statsmodels", version("statsmodels"))

linearmodels 7.0 | statsmodels 0.14.6


In [2]:
import pandas as pd
import numpy as np
from linearmodels.panel import PanelOLS

pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

from config import (DATA_DIR, OUT_DIR, INTERMEDIATE_DIR, FIG_DIR, SEED, TIER_MIN_WEEK, TIER_MAX_WEEK,
                    CAMP_MIN_WEEK, CAMP_MAX_WEEK, MID_WEEK, CELL_SIZE_THRESH, TIER_EN,
                    load_tiers, load_panel, load_camp_full)



## 1. Rebuild the pipeline (same as steps 1-4)

Exactly the same logic as the earlier steps. Nothing new: this builds the input data for step 6.

In [3]:
panel = load_panel()
tier_df = load_tiers()
print(f"Panel shape: {panel.shape}")
print(f"Households: {panel['household_key'].nunique():,} / weeks: {panel['WEEK_NO'].nunique()}")

Panel shape: (172500, 9)
Households: 2,500 / weeks: 69


## 2. [Re-check] Step 4 cell-size table: keep it next to the regression results when interpreting

In [4]:
def cell_size_table(tier_col):
    out = {}
    for t in ["TypeA", "TypeB", "TypeC"]:
        hh_active = panel.loc[panel[f"active_{t}"] == 1, ["household_key", tier_col]].drop_duplicates()
        out[t] = hh_active.groupby(tier_col, observed=True)["household_key"].nunique()
    return pd.DataFrame(out)

cell3 = cell_size_table("tier3")
print("[Type x 3-tier recipient households: always read next to the regression coefficients]")
print(cell3)
print(f"\nCombinations below {CELL_SIZE_THRESH} households: results are reference only")


[Type x 3-tier recipient households: always read next to the regression coefficients]
       TypeA  TypeB  TypeC
tier3                     
low      187     90     28
mid      554    305     74
high     772    628    295

Combinations below 30 households: results are reference only


## 3. Create the type x tier interaction dummies (9)

Each dummy means "this household belongs to this tier and this campaign type was active in this week".
The omitted reference state is "no type active that week". The tier main effect is absorbed by the household fixed effects
(tier is a time-invariant characteristic), so it is not entered separately.

In [5]:
for t in ["TypeA", "TypeB", "TypeC"]:
    for tier in ["low", "mid", "high"]:
        col = f"{t}_x_{tier}"
        panel[col] = ((panel[f"active_{t}"] == 1) & (panel["tier3"] == tier)).astype(int)

interaction_cols = [f"{t}_x_{tier}" for t in ["TypeA", "TypeB", "TypeC"] for tier in ["low", "mid", "high"]]
print("Interaction variables created:", interaction_cols)
print(panel[interaction_cols].sum().rename("active household-weeks"))


Interaction variables created: ['TypeA_x_low', 'TypeA_x_mid', 'TypeA_x_high', 'TypeB_x_low', 'TypeB_x_mid', 'TypeB_x_high', 'TypeC_x_low', 'TypeC_x_mid', 'TypeC_x_high']
TypeA_x_low      2166
TypeA_x_mid      9191
TypeA_x_high    17297
TypeB_x_low       787
TypeB_x_mid      2786
TypeB_x_high     9687
TypeC_x_low       329
TypeC_x_mid       996
TypeC_x_high     4463
Name: active household-weeks, dtype: int64


In [6]:
# ============================================================
# [Robustness check] split TypeC x high into "first exposure" vs "repeat exposure"
# ============================================================

def add_spell_number(df, type_col="active_TypeC"):
    """Group consecutive active weeks per household_key into one campaign spell and number the spells"""
    sub = df.loc[df[type_col] == 1, ["household_key", "WEEK_NO"]].sort_values(["household_key", "WEEK_NO"], kind="stable").copy()
    sub["is_new_spell"] = (
        (sub["household_key"] != sub["household_key"].shift())
        | (sub["WEEK_NO"] != sub["WEEK_NO"].shift() + 1)
    ).astype(int)
    sub["spell_no"] = sub.groupby("household_key")["is_new_spell"].cumsum()
    return sub[["household_key", "WEEK_NO", "spell_no"]]

spell_df = add_spell_number(panel, "active_TypeC")
panel = panel.merge(spell_df, on=["household_key", "WEEK_NO"], how="left")
panel["spell_no"] = panel["spell_no"].fillna(0).astype(int)  # 0 for weeks when TypeC is not active

panel["TypeC_x_high_first"] = (
    (panel["active_TypeC"] == 1) & (panel["tier3"] == "high") & (panel["spell_no"] == 1)
).astype(int)
panel["TypeC_x_high_repeat"] = (
    (panel["active_TypeC"] == 1) & (panel["tier3"] == "high") & (panel["spell_no"] >= 2)
).astype(int)

print("First-exposure household-weeks:", panel["TypeC_x_high_first"].sum(),
      "/ unique households:", panel.loc[panel["TypeC_x_high_first"]==1, "household_key"].nunique())
print("Repeat-exposure household-weeks:", panel["TypeC_x_high_repeat"].sum(),
      "/ unique households:", panel.loc[panel["TypeC_x_high_repeat"]==1, "household_key"].nunique())

# From the original list of 9, drop TypeC_x_high and replace it with the two split terms
interaction_cols_v2 = [c for c in interaction_cols if c != "TypeC_x_high"] + ["TypeC_x_high_first", "TypeC_x_high_repeat"]
print("\nFinal variable list:", interaction_cols_v2)

First-exposure household-weeks: 2855 / unique households: 295
Repeat-exposure household-weeks: 1608 / unique households: 131

Final variable list: ['TypeA_x_low', 'TypeA_x_mid', 'TypeA_x_high', 'TypeB_x_low', 'TypeB_x_mid', 'TypeB_x_high', 'TypeC_x_low', 'TypeC_x_mid', 'TypeC_x_high_first', 'TypeC_x_high_repeat']


## 4. Main regression: PanelOLS (household and week two-way fixed effects + household-clustered SE)

In [7]:
reg_df = panel.copy()
reg_df["entity"] = reg_df["household_key"]
reg_df["time"] = reg_df["WEEK_NO"]
reg_df = reg_df.set_index(["entity", "time"])

model = PanelOLS(
    dependent=reg_df["spend"],
    exog=reg_df[interaction_cols],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
result = model.fit(cov_type="clustered", cluster_entity=True)
print(result.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                  spend   R-squared:                        0.0006
Estimator:                   PanelOLS   R-squared (Between):             -0.0188
No. Observations:              172500   R-squared (Within):               0.0004
Date:                Wed, Oct 21 2026   R-squared (Overall):             -0.0106
Time:                        16:42:13   Log-likelihood                -9.015e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      11.544
Entities:                        2500   P-value                           0.0000
Avg Obs:                       69.000   Distribution:                F(9,169923)
Min Obs:                       69.000                                           
Max Obs:                       69.000   F-statistic (robust):             2.3131
                            

In [8]:
# --- source CSV: model diagnostics
pd.DataFrame([{
    "n_obs": int(result.nobs), "n_households": int(result.entity_info["total"]), "n_weeks": int(result.time_info["total"]),
    "r_squared_within": result.rsquared_within, "poolability_f": result.f_pooled.stat, "poolability_p": result.f_pooled.pval,
    "f_stat_nonrobust": result.f_statistic.stat, "f_p_nonrobust": result.f_statistic.pval,
    "f_stat_clustered": result.f_statistic_robust.stat, "f_p_clustered": result.f_statistic_robust.pval,
}]).to_csv(OUT_DIR / "fe_diagnostics.csv", index=False)

In [9]:
# ============================================================
# [Robustness check] regression with first vs repeat exposure split
# ============================================================
reg_df2 = panel.copy()
reg_df2["entity"] = reg_df2["household_key"]
reg_df2["time"] = reg_df2["WEEK_NO"]
reg_df2 = reg_df2.set_index(["entity", "time"])

model_v2 = PanelOLS(
    dependent=reg_df2["spend"],
    exog=reg_df2[interaction_cols_v2],
    entity_effects=True,
    time_effects=True,
    drop_absorbed=True
)
result_v2 = model_v2.fit(cov_type="clustered", cluster_entity=True)
print(result_v2.summary)

print()
print("="*50)
print("[Key comparison] first exposure vs repeat exposure")
print("="*50)
compare = pd.DataFrame({
    "coef": result_v2.params[["TypeC_x_high_first", "TypeC_x_high_repeat"]],
    "se": result_v2.std_errors[["TypeC_x_high_first", "TypeC_x_high_repeat"]],
    "p": result_v2.pvalues[["TypeC_x_high_first", "TypeC_x_high_repeat"]],
})
print(compare)
print(f"\n(For reference) original pooled coefficient: -6.2526, p=0.0004")

                          PanelOLS Estimation Summary                           
Dep. Variable:                  spend   R-squared:                        0.0007
Estimator:                   PanelOLS   R-squared (Between):             -0.0192
No. Observations:              172500   R-squared (Within):               0.0004
Date:                Wed, Oct 21 2026   R-squared (Overall):             -0.0108
Time:                        16:42:18   Log-likelihood                -9.015e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      11.171
Entities:                        2500   P-value                           0.0000
Avg Obs:                       69.000   Distribution:               F(10,169922)
Min Obs:                       69.000                                           
Max Obs:                       69.000   F-statistic (robust):             2.1319
                            

In [10]:
# --- source CSV: first vs repeat exposure (Type C x high)
_ci2 = result_v2.conf_int()
_rows = []
for _lab, _col in [("first", "TypeC_x_high_first"), ("repeat", "TypeC_x_high_repeat")]:
    _rows.append({"exposure": _lab, "coef": result_v2.params[_col], "se": result_v2.std_errors[_col],
                  "ci_low": _ci2.loc[_col, "lower"], "ci_high": _ci2.loc[_col, "upper"], "p_value": result_v2.pvalues[_col],
                  "n_households": panel.loc[panel[_col] == 1, "household_key"].nunique(),
                  "n_household_weeks": int(panel[_col].sum())})
pd.DataFrame(_rows).to_csv(OUT_DIR / "fe_first_repeat.csv", index=False)

### 4-1. Alternative: statsmodels (if linearmodels is not installed, for small-scale checks)

With many households the dummy matrix is large and can be slow. If linearmodels works normally, skip this cell.

In [11]:
# import statsmodels.api as sm
#
# reg_df2 = panel.reset_index(drop=True)
# hh_dummies = pd.get_dummies(reg_df2["household_key"], prefix="hh", drop_first=True)
# week_dummies = pd.get_dummies(reg_df2["WEEK_NO"], prefix="wk", drop_first=True)
# X = pd.concat([reg_df2[interaction_cols], hh_dummies, week_dummies], axis=1)
# X = sm.add_constant(X.astype(float))
# y = reg_df2["spend"].astype(float)
# ols_result = sm.OLS(y, X).fit(cov_type="cluster", cov_kwds={"groups": reg_df2["household_key"]})
# print(ols_result.summary())


## 5. Coefficient summary: arranged next to the cell sizes

Before moving on to the next steps (multiple-comparison correction, placebo test), arrange the results together with the cell sizes and the step 5 results.

In [12]:
coef_table = result.params.to_frame("coef")
coef_table["se"] = result.std_errors
coef_table["t"] = result.tstats
coef_table["p_raw"] = result.pvalues

# Split type/tier out of the variable name (TypeX_x_tier) and join with the cell-size table
coef_table = coef_table.reset_index().rename(columns={"index": "variable"})
coef_table[["Type", "tier"]] = coef_table["variable"].str.split("_x_", expand=True)

cell_long = cell3.stack().reset_index()
cell_long.columns = ["tier", "Type", "n_recipients"]

coef_table = coef_table.merge(cell_long, on=["Type", "tier"], how="left")
coef_table["small_cell"] = coef_table["n_recipients"] < CELL_SIZE_THRESH

_pt = pd.read_csv(OUT_DIR / "pretrend_tests.csv").set_index("scope")
pretrend_note = {t: f"step 5 p={_pt.loc[TIER_EN[t], 'p_value']:.3f} ({int(_pt.loc[TIER_EN[t], 'n_never_receive']):,} non-recipient households)" for t in ["low", "mid", "high"]}
coef_table["pretrend_note"] = coef_table["tier"].map(pretrend_note)

print(coef_table[["Type", "tier", "coef", "se", "p_raw", "n_recipients", "small_cell", "pretrend_note"]]
      .sort_values("coef", ascending=False, kind="stable").to_string(index=False))


 Type tier      coef       se    p_raw  n_recipients  small_cell                                 pretrend_note
TypeC  low  9.675478 6.939703 0.163253            28        True step 5 p=0.764 (624 non-recipient households)
TypeB  low  1.442018 2.335870 0.537014            90       False step 5 p=0.764 (624 non-recipient households)
TypeA  low  1.056302 1.431923 0.460710           187       False step 5 p=0.764 (624 non-recipient households)
TypeA  mid  0.819589 0.687613 0.233289           554       False step 5 p=0.053 (243 non-recipient households)
TypeB  mid  0.417724 1.362253 0.759116           305       False step 5 p=0.053 (243 non-recipient households)
TypeA high -0.162506 0.774218 0.833748           772       False  step 5 p=0.613 (49 non-recipient households)
TypeC  mid -1.219311 4.360564 0.779768            74       False step 5 p=0.053 (243 non-recipient households)
TypeB high -2.429732 0.965271 0.011832           628       False  step 5 p=0.613 (49 non-recipient households)
T

In [13]:
# --- source CSV: the nine campaign-type x tier coefficients (no multiple-testing adjustment yet)
_ci = result.conf_int()
_fe = coef_table[["Type", "tier", "coef", "se", "t", "p_raw", "n_recipients", "small_cell"]].copy()
_fe["ci_low"] = [_ci.loc[f"{r.Type}_x_{r.tier}", "lower"] for r in _fe.itertuples()]
_fe["ci_high"] = [_ci.loc[f"{r.Type}_x_{r.tier}", "upper"] for r in _fe.itertuples()]
_fe["tier"] = _fe["tier"].map(TIER_EN)
_fe.rename(columns={"Type": "type", "t": "t_stat"}) \
    [["type", "tier", "coef", "se", "ci_low", "ci_high", "t_stat", "p_raw", "n_recipients", "small_cell"]] \
    .to_csv(OUT_DIR / "fe_coefficients.csv", index=False)

## 6. Interpretation checklist (check before moving on)

1. Which type x tier coefficients are significant (p_raw < 0.05)?
2. Among them, is any flagged `small_cell=True` (low x TypeC)? If so, reference only
3. For significant results from tiers whose `pretrend_note` says "borderline" or "low power", state in the conclusion that
   the campaign effect and the pre-trend were not fully separated
4. **These are still raw p-values, before multiple-comparison correction**, so do not conclude "significant" here
   -> the final judgment comes in the next step (step 7, Benjamini-Hochberg correction)

## Next step
Use this result (coef_table) as is in step 7 (multiple-comparison correction) and step 8 (placebo test, 916 households that never received a campaign).

## 5. Lead-effect test: was spending already low in the window just before exposure?

**Purpose**: check the competing interpretation "the company sent Type C to high spenders whose spending was already falling" (reverse targeting) with **the same estimator** as the main regression.
- Sample: Type C x high households. Using each household's **first Type C active week (`first_start`)**, add `Lead4` (1-4 weeks before) and `Lead8` (5-8 weeks before) dummies to the main regression.
- Household and week fixed effects, household-clustered SE, and **the other type-active dummies are included as in the main regression**.
- The reading rule is recorded here and in `FINAL_NUMBERS_CASE2.md` ("Step 3: lead-effect test, reading rule"); this notebook does not establish that it was fixed before the test was first run. If the CI includes 0: "no evidence of a pre-decline"; if negative with a CI that excludes 0: "possible reverse targeting".

In [14]:

th = panel[(panel["active_TypeC"] == 1) & (panel["tier3"] == "high")]
first_start = th.groupby("household_key")["WEEK_NO"].min().rename("first_start")

pl = panel.copy()
fs = pl["household_key"].map(first_start)
wk = pl["WEEK_NO"]
pl["Lead4"] = (fs.notna() & (wk >= fs - 4) & (wk <= fs - 1)).astype(int)
pl["Lead8"] = (fs.notna() & (wk >= fs - 8) & (wk <= fs - 5)).astype(int)

# household-level facts: how much of each lead window lies inside the panel, and Type A/B exposure inside it
hh = first_start.to_frame()
hh["n_lead4_weeks_in_panel"] = (hh["first_start"] - CAMP_MIN_WEEK).clip(lower=0, upper=4)
hh["n_lead8_weeks_in_panel"] = (hh["first_start"] - CAMP_MIN_WEEK - 4).clip(lower=0, upper=4)
pl["_ab"] = (pl["active_TypeA"] == 1) | (pl["active_TypeB"] == 1)
hh["ab_in_lead4"] = pl[pl["Lead4"] == 1].groupby("household_key")["_ab"].any().reindex(hh.index).fillna(False)
hh["ab_in_lead8"] = pl[pl["Lead8"] == 1].groupby("household_key")["_ab"].any().reindex(hh.index).fillna(False)
hh.to_csv(INTERMEDIATE_DIR / "lead_effect_households.csv", index_label="household_key")

reg_lead = pl.set_index(["household_key", "WEEK_NO"])
exog_lead = interaction_cols + ["Lead4", "Lead8"]
model_lead = PanelOLS(dependent=reg_lead["spend"], exog=reg_lead[exog_lead],
                      entity_effects=True, time_effects=True, drop_absorbed=True)
res_lead = model_lead.fit(cov_type="clustered", cluster_entity=True)
print(res_lead.summary)


                          PanelOLS Estimation Summary                           
Dep. Variable:                  spend   R-squared:                        0.0008
Estimator:                   PanelOLS   R-squared (Between):             -0.0227
No. Observations:              172500   R-squared (Within):               0.0007
Date:                Wed, Oct 21 2026   R-squared (Overall):             -0.0127
Time:                        16:42:25   Log-likelihood                -9.015e+05
Cov. Estimator:             Clustered                                           
                                        F-statistic:                      13.135
Entities:                        2500   P-value                           0.0000
Avg Obs:                       69.000   Distribution:               F(11,169921)
Min Obs:                       69.000                                           
Max Obs:                       69.000   F-statistic (robust):             2.5412
                            

In [15]:

# --- source CSVs: lead-effect coefficients and household facts (reading rule as recorded in FINAL_NUMBERS_CASE2.md; order of writing vs. first run is not established)
def reading(coef, lo, hi):
    if lo <= 0 <= hi:
        return "no evidence of pre-decline (CI includes 0)"
    return "possible reverse targeting (negative, CI excludes 0)" if coef < 0 else "higher spending before exposure (positive, CI excludes 0)"

_ci = res_lead.conf_int()
_rows = []
for _term, _lab, _col in [("Lead4", "lead4 (1-4 weeks before first Type C week)", "Lead4"),
                          ("Lead8", "lead8 (5-8 weeks before first Type C week)", "Lead8"),
                          ("TypeC_x_high", "typeC_x_high (same model, leads included)", "TypeC_x_high")]:
    _r = {"term": _lab, "coef": res_lead.params[_term], "se": res_lead.std_errors[_term],
          "ci_low": _ci.loc[_term, "lower"], "ci_high": _ci.loc[_term, "upper"], "t_stat": res_lead.tstats[_term],
          "p_value": res_lead.pvalues[_term]}
    if _term.startswith("Lead"):
        _r["n_households_with_term"] = pl.loc[pl[_col] == 1, "household_key"].nunique()
        _r["n_household_weeks_with_term"] = int(pl[_col].sum())
        _r["reading"] = reading(_r["coef"], _r["ci_low"], _r["ci_high"])
    else:
        _r["n_households_with_term"] = pl.loc[pl[_col] == 1, "household_key"].nunique()
        _r["n_household_weeks_with_term"] = int(pl[_col].sum())
        _r["reading"] = "reference: main-model coefficient was " + f"{result.params[_term]:.4f}"
    _rows.append(_r)
pd.DataFrame(_rows).to_csv(OUT_DIR / "lead_effect_coefficients.csv", index=False)

_k = len(res_lead.params)
_R = np.zeros((2, _k))
_R[0, list(res_lead.params.index).index("Lead4")] = 1
_R[1, list(res_lead.params.index).index("Lead8")] = 1
_wald = res_lead.wald_test(restriction=_R, value=np.zeros(2))
_ab4_weeks = pl.loc[pl["Lead4"] == 1, "_ab"].mean() * 100
_ab8_weeks = pl.loc[pl["Lead8"] == 1, "_ab"].mean() * 100
pd.DataFrame([{
    "n_typeC_high_households": len(hh),
    "first_start_min": hh["first_start"].min(), "first_start_median": hh["first_start"].median(), "first_start_max": hh["first_start"].max(),
    "n_lead4_truncated_by_panel_start": int((hh["n_lead4_weeks_in_panel"] < 4).sum()),
    "n_lead8_truncated_by_panel_start": int((hh["n_lead8_weeks_in_panel"] < 4).sum()),
    "n_lead8_no_weeks_in_panel": int((hh["n_lead8_weeks_in_panel"] == 0).sum()),
    "n_households_with_typeAB_in_lead4": int(hh["ab_in_lead4"].sum()),
    "n_households_with_typeAB_in_lead8": int(hh["ab_in_lead8"].sum()),
    "n_households_with_typeAB_in_either": int((hh["ab_in_lead4"] | hh["ab_in_lead8"]).sum()),
    "pct_lead4_household_weeks_with_typeAB": _ab4_weeks, "pct_lead8_household_weeks_with_typeAB": _ab8_weeks,
    "joint_wald_stat_lead4_lead8": _wald.stat, "joint_wald_p": _wald.pval,
    "n_obs": int(res_lead.nobs), "r_squared_within": res_lead.rsquared_within,
}]).to_csv(OUT_DIR / "lead_effect_summary.csv", index=False)
